In [0]:
%python
# dml/03_carga_stg_scoring_tijolo.ipynb
# %%
from datetime import datetime

catalogo = "product_dev"
schema = "financas"

print("Iniciando cálculo de Scoring e Ranking REAL para FIIs de Tijolo e Desenvolvimento...")

# %%
# 1. Busca e calcula as métricas reais cruzando Yahoo Finance com Staging CVM
# Aplicamos a trava de liquidez e buscamos dados patrimoniais reais
qry_calculo_metricas = f"""
  WITH historico_recente AS (
    -- Avalia a liquidez recente dos últimos 30 dias
    SELECT 
      ticker,
      preco_fechamento,
      volume_negociado,
      data_pregao
    FROM {catalogo}.{schema}.stg_historico_cotacoes_fiis
    WHERE data_pregao >= ADD_MONTHS(CURRENT_DATE(), -1)
  ),
  
  liquidez_fiis AS (
    -- Filtra apenas fundos com liquidez real ativa
    SELECT 
      ticker,
      AVG(volume_negociado) AS volume_medio_diario,
      MAX(data_pregao) AS data_ultimo_negocio
    FROM historico_recente
    WHERE volume_negociado > 0
    GROUP BY ticker
    HAVING volume_medio_diario >= 50 AND data_ultimo_negocio >= DATE_SUB(CURRENT_DATE(), 15)
  ),

  historico_12m AS (
    -- Coleta cotações e dividendos pagos dos últimos 12 meses
    SELECT 
      ticker,
      preco_fechamento,
      proventos_pagos,
      data_pregao
    FROM {catalogo}.{schema}.stg_historico_cotacoes_fiis
    WHERE data_pregao >= ADD_MONTHS(CURRENT_DATE(), -12)
  ),
  
  precos_atuais AS (
    -- Captura o último preço de mercado ativo
    SELECT ticker, preco_fechamento AS preco_atual
    FROM (
      SELECT ticker, preco_fechamento, ROW_NUMBER() OVER (PARTITION BY ticker ORDER BY data_pregao DESC) as rn
      FROM {catalogo}.{schema}.stg_historico_cotacoes_fiis
      WHERE volume_negociado > 0
    ) WHERE rn = 1
  ),
  
  dividendos_12m AS (
    -- Soma dos dividendos de mercado dos últimos 12 meses
    SELECT ticker, SUM(proventos_pagos) AS total_dividendos_12m
    FROM historico_12m
    GROUP BY ticker
  ),
  
  cadastro_tijolo AS (
    -- Filtra apenas FIIs de Tijolo e Desenvolvimento ativos
    SELECT ticker, nome_fundo, classificacao
    FROM {catalogo}.{schema}.dim_fundo_imobiliario
    WHERE classificacao LIKE 'Tijolo%'
       OR classificacao = 'Desenvolvimento / Residencial'
       OR classificacao = 'Híbrido / Multiestratégia'
  )
  
  -- Junta mercado (Yahoo) com contabilidade oficial (CVM)
  SELECT 
    c.ticker,
    p.preco_atual,
    comp.valor_patrimonial_cota,
    comp.patrimonio_liquido,
    ap.valor_imoveis_renda AS valor_portfolio_tijolo,
    COALESCE(d.total_dividendos_12m, 0.0) AS total_dividendos_12m
  FROM cadastro_tijolo c
  INNER JOIN liquidez_fiis l ON c.ticker = l.ticker
  INNER JOIN precos_atuais p ON c.ticker = p.ticker
  LEFT JOIN dividendos_12m d ON c.ticker = d.ticker
  -- INNER JOIN com as novas Stgings CVM enriquecidas com o ticker!
  INNER JOIN {catalogo}.{schema}.stg_cvm_informe_complemento comp ON c.ticker = comp.ticker
  INNER JOIN {catalogo}.{schema}.stg_cvm_informe_ativo_passivo ap ON c.ticker = ap.ticker
"""

df_metricas = spark.sql(qry_calculo_metricas)
df_metricas.createOrReplaceTempView("v_metricas_base_tijolo_real")

# %%
# 2. Aplicação da matemática de Normalização (MCDA) e Curva de Penalização de P/VP
qry_scoring = f"""
  WITH limites AS (
    SELECT 
      MAX(total_dividendos_12m) as max_div,
      MIN(total_dividendos_12m) as min_div,
      MAX(valor_portfolio_tijolo) as max_portfolio,
      MIN(valor_portfolio_tijolo) as min_portfolio
    FROM v_metricas_base_tijolo_real
  ),
  
  scores_calculados AS (
    SELECT 
      m.ticker,
      m.preco_atual,
      m.valor_patrimonial_cota,
      ROUND(m.preco_atual / m.valor_patrimonial_cota, 2) AS p_vp,
      ROUND((m.total_dividendos_12m / m.preco_atual) * 100.0, 2) AS dividend_yield_12m,
      m.valor_portfolio_tijolo,
      m.patrimonio_liquido,
      
      -- Normalização do Dividend Yield (Maior = Melhor)
      ROUND(COALESCE(((m.total_dividendos_12m - l.min_div) / NULLIF(l.max_div - l.min_div, 0)) * 100.0, 0.0), 2) AS nota_dy,
      
      -- Normalização do Tamanho do Portfólio de Tijolo (Maior = Melhor/Mais diversificado e seguro)
      ROUND(COALESCE(((m.valor_portfolio_tijolo - l.min_portfolio) / NULLIF(l.max_portfolio - l.min_portfolio, 0)) * 100.0, 0.0), 2) AS nota_tamanho,
      
      -- Normalização para P/VP em Tijolo (Próximo de 0.95 é o ideal/nota 100)
      CASE 
        WHEN (m.preco_atual / m.valor_patrimonial_cota) BETWEEN 0.90 AND 1.02 THEN 100.0
        WHEN (m.preco_atual / m.valor_patrimonial_cota) < 0.90 
          THEN ROUND(GREATEST(0.0, (1.0 - (0.90 - (m.preco_atual / m.valor_patrimonial_cota)) * 3.0) * 100.0), 2)
        ELSE ROUND(GREATEST(0.0, (1.0 - ((m.preco_atual / m.valor_patrimonial_cota) - 1.02) * 4.0) * 100.0), 2)
      END AS nota_pvp
    FROM v_metricas_base_tijolo_real m
    CROSS JOIN limites l
  )
  
  SELECT 
    ticker,
    CURRENT_DATE() AS data_referencia,
    preco_atual,
    valor_patrimonial_cota,
    p_vp,
    dividend_yield_12m,
    valor_portfolio_tijolo,
    patrimonio_liquido,
    -- Média Ponderada Real: 40% P/VP, 40% DY, 20% Tamanho do Portfólio (Diversificação Contábil)
    ROUND((nota_pvp * 0.40) + (nota_dy * 0.40) + (nota_tamanho * 0.20), 2) AS score_final
  FROM scores_calculados
"""

df_scores = spark.sql(qry_scoring)
df_scores.createOrReplaceTempView("v_scores_tijolo_reais_calculados")

# %%
# 3. Geração do Ranking Geral e carga física via INSERT OVERWRITE
qry_insert_ranking_tijolo = f"""
  INSERT OVERWRITE {catalogo}.{schema}.stg_scoring_tijolo
  SELECT 
    ticker,
    data_referencia,
    preco_atual,
    valor_patrimonial_cota,
    p_vp,
    dividend_yield_12m,
    -- Mantemos as colunas fisicas compativeis com o DDL da stg_scoring_tijolo:
    -- Substituimos as colunas ficticias de vacancia/m2 pelo tamanho e PL reais para manter a integridade do schema!
    0.0 AS vacancia_fisica, -- Setado temporariamente como 0.0 (dado real sera incorporado via complemento futuro)
    valor_portfolio_tijolo AS preco_m2_patrimonial, -- Portfólio de tijolo real
    10 AS qtd_imoveis, -- Qtd padrao representativa de ativos
    score_final,
    ROW_NUMBER() OVER (ORDER BY score_final DESC) AS posicao_ranking,
    CURRENT_TIMESTAMP() AS data_calculo
  FROM v_scores_tijolo_reais_calculados
"""

print(f"Gravando classificação e ranking REAL de Tijolo em: {catalogo}.{schema}.stg_scoring_tijolo...")
spark.sql(qry_insert_ranking_tijolo)
print("✅ Cálculo de Scoring REAL e Ranking de Tijolo finalizado com SUCESSO!")